# 예외 항목 데이터 전처리 및 RAG 연결


- 기존 임베딩한 모델 로드 및 기존 벡터 저장소 불러오기
- 이후 text_splitter 다시 정의해서 청킹 사이즈 정하기


In [ ]:
# 1. 임베딩 모델 다시 로드
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}
)

# 2. 기존 벡터 저장소 불러오기 (재임베딩 불필요)
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)

# 3. text_splitter 다시 정의
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)


In [ ]:
from tricare_exclusions_crawler import crawl_exclusions, to_documents

results = crawl_exclusions()
exclusion_docs = to_documents(results)


In [ ]:
# 기존 text_splitter 재사용
exclusion_chunks = text_splitter.split_documents(exclusion_docs)

print(f'🔪 제외 항목 청크 수: {len(exclusion_chunks)}개')


In [ ]:
# 기존 vector_store에 추가
vector_store.add_documents(exclusion_chunks)
print(f'✅ 추가 후 벡터 수: {vector_store._collection.count()}개')


---


# 기존 DEMO 1, 2, 3 모델 로드하기
1. 환경 변수 로드
2. 임베딩 모델 로드(벡터 저장소 불러올 때 필요함)
3. 기존 벡터 저장소 불러오기
4. retriever 재정의하기
5. LLM 재정의하기
6. 언어 감지 함수 및 언어 맵 정의
7. format_docs + make_rag_chain 재조립
=> 결과 확인


In [ ]:
# 환경 변수 로드
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# 임베딩 모델 로드
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}
)


In [ ]:
# 기존 벡터 저장소 불러오기
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)


In [ ]:
# retriever 재정의
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6}
)
table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)


In [ ]:
# LLM 재정의
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)


In [ ]:
#!pip install langchain-teddynote


In [ ]:
# 언어 감지 함수 및 언어 맵 정의
# tricare_rag_pipeline.ipynb와 동일하게 유지 (출력 형식 통일)
# 사용자 질문 언어를 감지해 프롬프트에 answer_language를 주입하기 위해 필요
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': 'the same language as the user\'s question',
}


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# format_docs: 소스와 페이지 정보를 포함한 컨텍스트 문자열 생성
# tricare_rag_pipeline.ipynb와 동일 구현으로 통일
# 기존에 '...'(미완성)으로 남아있던 함수 본문을 완성
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '')
        label = f'[{source}{f", p.{page+1}" if page else ""}]'
        result.append(f'{label}{doc.page_content}')
    return '\n\n'.join(result)


# make_rag_chain: 질문 언어를 감지해 프롬프트에 answer_language를 f-string으로 주입
# retriever를 인자로 받아 텍스트용(retriever)·표 전용(table_retriever) 모두 사용 가능
def make_rag_chain(question: str, retriever, model):
    language_code = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, say "해당 내용은 제공된 문서에서 확인되지 않습니다." '
        'in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official document names or insurance terms.\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )


In [ ]:
print('✅ 준비 완료')


---


# DEMO 1, 2, 3 사용


In [ ]:
# 질문 설정
question = '보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인가요?'
# question = 'I plan on still working when I turn 65. According to Medicare, I can wait to sign up for Medicare Part B since I'll continue to have health insurance through my employer. How does this affect my TRICARE?'
# question = 'What is the TRICARE Prime Demo?'


In [ ]:
# Demo 2 — RAG (텍스트 + CSV + 제외 항목)
# 팀원 출력 형식 통일: 'QUESTION:' + '=== ANSWER ===' + '=== RETRIEVED DOCS ===' 구조
print('=' * 100)
print('QUESTION:', question)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    print(f'  [{i+1}] {src}: {chunk.page_content[:80]}...')

# 질문 언어 감지 후 체인 생성
rag_chain = make_rag_chain(question, retriever, model)
answer = rag_chain.invoke(question)

print('=== ANSWER ===')
print(answer)

print('\n=== RETRIEVED DOCS ===')
for doc in retrieved:
    print(doc.metadata)


**사용 방법 요약**
```
질문 바꾸고 싶을 때  → question 변수만 수정
Demo 2만 쓰고 싶을 때 → Demo 2 셀만 실행
Demo 2, 3 비교하고 싶을 때 → 두 셀 순서대로 실행
```


In [ ]:
# Demo 3 — RAG (표 전용)
# 팀원 출력 형식 통일: 'QUESTION:' + '=== ANSWER ===' + '=== RETRIEVED DOCS ===' 구조
print('=' * 100)
print('QUESTION:', question)

table_rag_chain = make_rag_chain(question, table_retriever, model)
answer = table_rag_chain.invoke(question)

print('=== ANSWER ===')
print(answer)

print('\n=== RETRIEVED DOCS ===')
for doc in table_retriever.invoke(question):
    print(doc.metadata)
